### Imports

In [1]:
import json
import time
from pathlib import Path

import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.config import CONFIG
from src.lstm_utils.lstm_dataset import LSTMDataset
from src.lstm_utils.lstm_tokeniser import LSTMTokeniser
from src.lstm_utils.lstm_training import train_one_epoch, validate, evaluate
from src.lstm_utils.lstm_tuning import run_hyperparameter_sweep
from src.models.lstm_classifier import LSTMClassifier

/Users/jackrong/University/34812-cwk-S-Project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Constants

In [2]:
TRAIN_DATA_PATH = Path(f'../data/train.csv')
VAL_DATA_PATH = Path(f'../data/dev.csv')
BENCHMARK_DATA_PATH = Path(f'../data/NLI_trial.csv')

HYPERPARAMETERS_PATH = Path(f'../hyperparameters/lstm.json')
MODEL_PATH = Path(f'../models/lstm.pt')

### Device

In [3]:
device = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps' if torch.backends.mps.is_available() else
    'cpu'
)
print(f'Using device: {device}')

Using device: mps


### Set a constant seed for reproducability

In [4]:
generator = torch.manual_seed(CONFIG.seed)

### Verify tokenising works

In [5]:
def verify_tokeniser() -> None:
    train_df = pd.read_csv(TRAIN_DATA_PATH)
    sample = train_df.sample(1).iloc[0]
    sample_premise = sample['premise']

    lstm_tokeniser = LSTMTokeniser()
    premise_ids, premise_mask = lstm_tokeniser.encode(sample_premise, CONFIG.lstm.max_length)
    print(f'Sample premise: {sample_premise}')
    print(f'Premise ids: {premise_ids}')
    print(f'Premise mask: {premise_mask}')
    assert(len(premise_ids) == CONFIG.lstm.max_length)

# verify_tokeniser()

### Hyperparameter Tuning

In [6]:
if CONFIG.lstm.hyperparameter_tuning.should_run:
    lstm_tokeniser = LSTMTokeniser()

    train_dataset = LSTMDataset(TRAIN_DATA_PATH, lstm_tokeniser)
    val_dataset = LSTMDataset(VAL_DATA_PATH, lstm_tokeniser)

    train_dataloader = DataLoader(train_dataset, generator=generator, batch_size=CONFIG.batch_size, shuffle=True, drop_last=True)
    val_dataloader = DataLoader(val_dataset, generator=generator, batch_size=CONFIG.batch_size, shuffle=False)

    run_hyperparameter_sweep(device, lstm_tokeniser, train_dataloader, val_dataloader, HYPERPARAMETERS_PATH)

### Training Loop

In [7]:
def run_train() -> None:
    lstm_tokeniser = LSTMTokeniser()
    train_dataset = LSTMDataset(TRAIN_DATA_PATH, lstm_tokeniser)
    val_dataset = LSTMDataset(VAL_DATA_PATH, lstm_tokeniser)

    train_dataloader = DataLoader(train_dataset, generator=generator, batch_size=CONFIG.batch_size, shuffle=True, drop_last=True)
    val_dataloader = DataLoader(val_dataset, generator=generator, batch_size=CONFIG.batch_size, shuffle=False)

    hyperparameters = json.load(open(HYPERPARAMETERS_PATH, 'r'))
    print(f'Hyperparameters used: {hyperparameters}')
    print()

    model = LSTMClassifier(lstm_tokeniser, dropout=hyperparameters['dropout']).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=hyperparameters['lr'], weight_decay=hyperparameters['weight_decay'])
    criterion = nn.CrossEntropyLoss()

    best_acc = 0.0
    patience = 0
    train_losses, train_accs = [], []
    val_losses, val_accs = [], []
    total_start = time.time()
    for epoch in range(CONFIG.epochs):
        epoch_start = time.time()
        print(f'[Epoch {epoch + 1}/{CONFIG.epochs}]')

        train_loss, train_acc = train_one_epoch(device, model, criterion, optimizer, train_dataloader)
        train_losses.append(train_loss)
        train_accs.append(train_acc)

        val_loss, val_acc = validate(device, model, criterion, val_dataloader)
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        if val_acc > best_acc:
            best_acc = val_acc
            # Avoid saving GloVe embedding
            state = {k: v for k, v in model.state_dict().items() if 'embedding' not in k}
            torch.save(state, MODEL_PATH)
            patience = 0
        else:
            patience += 1

        epoch_time = time.time() - epoch_start
        elapsed = time.time() - total_start
        print(f'Train Loss: {train_loss:.2f} | Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.2f} | Val Acc: {val_acc:.2f}%')
        print(f'Epoch time: {epoch_time:.1f}s | Total elapsed: {elapsed:.1f}s')
        print()

        if patience == CONFIG.patience:
            print(f'Finishing training early, no improvement in {CONFIG.patience} epochs')
            break

    total_time = time.time() - total_start
    print(f'Training complete in {total_time:.1f}s')
    print(f'Best model had an accuracy of {best_acc:.2f}%.')

# train()

### Evaluation

In [8]:
def run_evaluate() -> None:
    print('Running final evaluation on benchmark...')

    lstm_tokeniser = LSTMTokeniser()
    model = LSTMClassifier(lstm_tokeniser, dropout=0.5).to(device)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device), strict=False)

    test_dataset = LSTMDataset(BENCHMARK_DATA_PATH, lstm_tokeniser)
    test_dataloader = DataLoader(test_dataset, generator=generator, batch_size=CONFIG.batch_size, shuffle=False)
    test_results = evaluate(device, model, test_dataloader)

    print('[Benchmark Results]')
    print(f'Accuracy:           {test_results["accuracy"] * 100:.2f}%')
    print(f'Macro Precision:    {test_results["macro_precision"]:.4f}')
    print(f'Macro Recall:       {test_results["macro_recall"]:.4f}')
    print(f'Macro F1:           {test_results["macro_f1"]:.4f}')
    print(f'Weighted Precision: {test_results["weighted_precision"]:.4f}')
    print(f'Weighted Recall:    {test_results["weighted_recall"]:.4f}')
    print(f'Weighted F1:        {test_results["weighted_f1"]:.4f}')

run_evaluate()

Running final evaluation on benchmark...
[Benchmark Results]
Accuracy:           84.00%
Macro Precision:    0.8390
Macro Recall:       0.8390
Macro F1:           0.8390
Weighted Precision: 0.8400
Weighted Recall:    0.8400
Weighted F1:        0.8400
